## LAB511: Create advanced Postgres-powered agentic apps with Azure HorizonDB

### Part 3.1: Introduction

**Welcome to the LAB511 Agent App Notebook!**

In this notebook, you will build a Agent Framework-based Agent that can reason over our database of legal cases you deployed in the previous steps. You will also incorporate external web service data, and use memory to improve its responses over time.

Agent Framework is an open-source SDK developed by Microsoft that helps developers create advanced AI agents by combining:

- LLMs (Large Language Models) like OpenAI's GPT models
- Agent Function Tools (sometimes called 'AI Function Tools', custom functions the agent can call)
- Memory (saving and recalling past conversations or facts)

An Agent in Agent Framework is a smart assistant that can:

- Respond to user prompts
- Decide which function tools to call
- Use external knowledge sources like databases or APIs
- Build better, grounded answers by combining model reasoning with real-world data

You are about to connect powerful components:

- Azure OpenAI (for embeddings and LLM chat completions)
- Azure PostgreSQL with Vector and Graph extensions (for fast semantic and graph search)
- APIs for real-world data (historical weather evidence)

As you progress, each section of code will incrementally build up these capabilities, and by the final step, you’ll have a highly capable legal research assistant.

**Architecture Diagram**

> The app we are going to build today:

### Part 3.2: Setup the Agent App Python imports

> **Note:** In your lab environment, we already have the PIP packages pre-deployed that are needed by the import statements in the following code block, so you do not need to perform any installs.

##### 🧠 *Technical Background Notes*

This set of imports prepares the technical foundation for building an AI-powered agent that interacts with a PostgreSQL database and Azure OpenAI services. `nest_asyncio` is used to allow nested event loops, which is essential when running asynchronous code inside a Jupyter notebook. Core Python modules like `os`, `asyncio`, `uuid`, and `requests` handle environment access, asynchronous execution, unique ID generation, and external API calls, respectively. `psycopg` provides modern database connectivity to PostgreSQL, while `pydantic` offers structured data validation and modeling through its `Field` component. The `Annotated` type from `typing` enables detailed parameter annotations for function tools. Finally, `dotenv` with `load_dotenv` allows secure loading of environment variables from a `.env` file.

The Agent Framework libraries enable creation of intelligent agents through `AzureOpenAIChatClient` for Azure OpenAI integration, `ChatAgent` for agent orchestration, `ChatMessageStore` for conversation memory, and `ai_function` decorator for registering callable tools. The framework also provides message handling components (`ChatMessage`, `TextContent`, `Role`) for managing conversation state and structure.

##### 📝 *Coding Tasks*

1. Run the cell below using the "▶" icon next to the cell.

1. This will run the code and show the output below.  Since these are just imports, there is nothing to show at the end other than a check mark showing success.

    > **Note:** The first time this code block is ran, it may take around 10-15 seconds.

In [ ]:
# Allow nested event loops (needed so async agent calls work inside Jupyter)
import nest_asyncio
nest_asyncio.apply()

# Standard library
import os
import re
import sys
import io
import json
import time
import asyncio
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Type hints used by tool signatures (Annotated drives the JSON schema the agent sees)
from typing import Annotated, Optional, List, Literal

# Third-party: HTTP, data validation, database driver, env loading
import requests
from pydantic import Field
import psycopg
from dotenv import load_dotenv

# UI + notebook display
import gradio as gr
from IPython.display import Markdown


In [ ]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT   = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
AZURE_OPENAI_KEY        = os.environ["AZURE_OPENAI_KEY"]
AZURE_EMBED_DEPLOYMENT  = os.environ["AZURE_EMBED_DEPLOYMENT"]
AZURE_API_VERSION       = os.environ["AZURE_API_VERSION"]

DB_CONFIG = {
    "host":     os.environ["AZURE_PG_HOST"],
    "dbname":   os.environ["AZURE_PG_NAME"],
    "user":     os.environ["AZURE_PG_USER"],
    "password": os.environ["AZURE_PG_PASSWORD"],
    "port":     os.environ["AZURE_PG_PORT"],
    "sslmode":  os.environ.get("AZURE_PG_SSLMODE", "require"),
}

print(AZURE_OPENAI_ENDPOINT)
print(AZURE_OPENAI_DEPLOYMENT)
print(AZURE_OPENAI_KEY)
print(AZURE_EMBED_DEPLOYMENT)
print(AZURE_API_VERSION)
print(DB_CONFIG)

## Tool 1

In [ ]:
@tool(description=(
    "BM25 full-text search over Washington case law opinions. "
    "Supports phrase proximity ('common enemy'~3), boolean operators "
    "(AND, OR, NOT), and optional filters by court level and minimum decision date. "
    "Returns the top matching cases ranked by BM25 relevance score. "
    "Use this first to surface canonical precedents by doctrine name or citation."
))
def keyword_case_search(
    query: Annotated[str, Field(
        description=(
            "BM25/Tantivy-syntax query over case opinions. "
            "Examples: 'common enemy doctrine', "
            "'\"surface water\"~3 AND (regrade OR grading)', "
            "'qualified immunity NOT prison'."
        )
    )],
    court_level: Annotated[Optional[str], Field(
        default=None,
        description=(
            "Optional filter: 'Washington Supreme Court' "
            "or 'Washington Court of Appeals'. "
            "None = both courts."
        ),
    )] = None,
    min_year: Annotated[Optional[int], Field(
        default=None,
        description="Optional filter: only return cases decided on or after this year."
    )] = None,
) -> str:
    print("keyword_case_search was called")
    print(f"  query={query!r}  court_level={court_level}  min_year={min_year}  limit=5")

    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        # Warm up: calling any pgfts function forces PostgreSQL to load the
        # pg_fts library, which runs _PG_init() and registers the planner hook
        # needed by fts_query() / fts_score().
        cur.execute("SELECT pgfts.hello_pg_fts();")
        cur.execute("SET search_path = public, pgfts;")

        # Build WHERE clauses
        where = ["fts_query(%s, 'idx_cases_fts')"]
        params = [query]
        if court_level:
            where.append("court_level = %s")
            params.append(court_level)
        if min_year:
            where.append("decision_date >= %s")
            params.append(f"{min_year}-01-01")

        cur.execute(
            f"""
            SELECT id, name, decision_date, court_level,
                   fts_score(opinion) AS score, opinion
            FROM cases
            WHERE {' AND '.join(where)}
            LIMIT 5;
            """,
            tuple(params),
        )
        rows = cur.fetchall()

    if not rows:
        return "No matches"

    lines = []
    for case_id, name, decision_date, level, score, opinion in rows:
        snippet = (opinion or "").replace("\n", " ")[:300]
        lines.append(
            f"{case_id} | {decision_date} | {level} | score={score:.3f} | "
            f"{name}: {snippet}..."
        )
    return "\n".join(lines)


In [ ]:
with psycopg.connect(**DB_CONFIG) as conn, conn.cursor() as cur:
    cur.execute("SHOW shared_preload_libraries;")
    libs = cur.fetchone()[0]
    print("shared_preload_libraries:")
    for lib in libs.split(","):
        print(f"  - {lib.strip()}")

In [ ]:
print("// Test Call of keyword_case_search //")
print(keyword_case_search("surface water OR flooding OR drainage"))

In [ ]:
client2 = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

agent2 = client2.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, why each is relevant, "
        "and a short one-sentence summary of the opinion. Run each tool only once."
    ),
    tools=[keyword_case_search],
)

user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the "
    "road."
)

print("// Functions the Agent Called: //")
result = await agent2.run(user_query)

print("")
print("// Agent Response: //")
Markdown(result.text)

## Tool 2

In [ ]:
VALID_COURT_LEVELS = {"Washington Supreme Court", "Washington Court of Appeals"}

@tool(description=(
    "Semantic vector search over Washington case opinions using DiskANN. "
    "Uses the precomputed opinions_vector column and supports metadata filtering "
    "by court_level and decision_date. With DiskANN Advanced Filtering enabled, "
    "filters are applied during index traversal for faster filtered vector search."
))
def semantic_case_search(
    query_text: Annotated[str, Field(
        description="Natural language query describing the fact pattern or legal issue."
    )],
    court_level: Annotated[Optional[str], Field(
        default=None,
        description=(
            "Optional filter: 'Washington Supreme Court' or 'Washington Court of Appeals'. "
            "None = both courts."
        ),
    )] = None,
    min_year: Annotated[Optional[int], Field(
        default=None,
        description="Optional filter: only return cases decided on or after this year."
    )] = None,
    use_advanced_filtering: Annotated[bool, Field(
        default=True,
        description="Enable DiskANN advanced filtering session GUCs (recommended)."
    )] = True,
    seed_case_ids: Annotated[Optional[List[int]], Field(
        default=None,
        description=(
            "Optional list of case ids (usually from keyword_case_search) to keep in view. "
            "If provided, any missing ids will be appended to the output."
        ),
    )] = None,
) -> str:
    print("semantic_case_search was called")
    print(
        f"  query_text={query_text!r}  court_level={court_level}  "
        f"min_year={min_year}  limit=5  advanced_filtering={use_advanced_filtering}  "
        f"seed_case_ids={'None' if not seed_case_ids else len(seed_case_ids)}"
    )

    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        # Enable DiskANN advanced filtering (session-level) per HorizonDB private preview guidance
        # This applies WHERE predicates during DiskANN graph traversal.
        if use_advanced_filtering:
            cur.execute("SET diskann.enable_filter_hook = 'true';")
            cur.execute("SET diskann.selectivity_min = '0.0';")
            cur.execute("SET diskann.selectivity_threshold = '1.0';")
            cur.execute("SET diskann.filtering_beta = 0.85;")
            cur.execute("SET diskann.l_value_is = 300;")
        else:
            cur.execute("SET diskann.enable_filter_hook = 'false';")

        # Build WHERE clause
        where = ["opinions_vector IS NOT NULL"]
        params = {"q": query_text, "lim": 5}

        if court_level:
            if court_level not in VALID_COURT_LEVELS:
                return f"Invalid court_level {court_level!r}. Use one of: {sorted(VALID_COURT_LEVELS)}"
            where.append("court_level = %(court_level)s")
            params["court_level"] = court_level

        if min_year:
            where.append("decision_date >= make_date(%(min_year)s, 1, 1)")
            params["min_year"] = min_year

        # Vector search uses the managed embedding model alias configured by AI Model Management
        # or BYOM model_registry.model_add. In HorizonDB, default-embedding
        # maps to text-embedding-3-small when managed models are enabled.
        cur.execute(
            f"""
            SELECT id, name, decision_date, court_level,
                   opinions_vector <=> azure_openai.create_embeddings('default-embedding', %(q)s)::vector AS distance,
                   opinion
            FROM public.cases
            WHERE {' AND '.join(where)}
            ORDER BY distance ASC
            LIMIT %(lim)s;
            """,
            params,
        )
        rows = cur.fetchall()

        # If provided, pull any seed ids not already returned (keeps Tool1 anchors visible)
        seed_rows = []
        if seed_case_ids:
            returned_ids = {r[0] for r in rows}
            missing = [i for i in seed_case_ids if i not in returned_ids]
            if missing:
                cur.execute(
                    """
                    SELECT id, name, decision_date, court_level,
                           NULL::float8 AS distance,
                           opinion
                    FROM public.cases
                    WHERE id = ANY(%s);
                    """,
                    (missing,),
                )
                seed_rows = cur.fetchall()

    if not rows and not seed_rows:
        return "No matches"

    lines = []
    for case_id, name, decision_date, level, dist, opinion in rows:
        snippet = (opinion or "").replace("\n", " ")[:300]
        dist_str = "n/a" if dist is None else f"{dist:.4f}"
        lines.append(
            f"{case_id} | {decision_date} | {level} | distance={dist_str} | "
            f"{name}: {snippet}..."
        )

    if seed_rows:
        lines.append("")
        lines.append("Seed cases (from keyword search):")
        for case_id, name, decision_date, level, _, opinion in seed_rows:
            snippet = (opinion or "").replace("\n", " ")[:200]
            lines.append(f"{case_id} | {decision_date} | {level} | {name}: {snippet}...")

    return "\n".join(lines)


## Test Tool 2 directly

In [ ]:
print("// Step 1: Keyword anchors (BM25) //")
kw = keyword_case_search(
    "'common enemy'~3 AND (surface water OR drainage OR stormwater)",
    court_level="Washington Supreme Court",
    min_year=1900,
)
print(kw)

# Pull ids from the formatted output (each line starts with "<id> | ...")
seed_ids = []
for line in kw.splitlines():
    m = re.match(r"^(\d+)\s+\|", line)
    if m:
        seed_ids.append(int(m.group(1)))

print("")
print("// Step 2: Semantic expansion (DiskANN) using flagship prompt //")
flagship_prompt = (
    "I have a client whose property keeps flooding after the developer next door "
    "regraded their lot and the city redid the drainage on the road."
)
sem = semantic_case_search(
    flagship_prompt,
    court_level=None,      # search both courts for applied authority
    min_year=1900,
    use_advanced_filtering=True,
    seed_case_ids=seed_ids
)
print(sem)

print("")
print("// Step 3 preview: seeds for the precedent graph tool //")
# Tool 3 will consume the union of ids from Tool 1 + Tool 2. For now, show the union list.
# (In the Tool 3 section, you'll pass this list into the AGE traversal query.)
print(sorted(set(seed_ids)))

## Test Tool 2 by re-assembling our Agent and running the same prompt

In [ ]:
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, why each is relevant, "
        "and a short one-sentence summary of the opinion. Run each tool only once."
    ),
    tools=[keyword_case_search,semantic_case_search]
)

user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the "
    "road."
)

print("// Functions the Agent Called: //")
result = await agent.run(user_query)

print("")
print("// Agent Response: //")
Markdown(result.text)

## Tool 3 - Graph

In [ ]:
@tool(description=(
    "Traverse the Washington case citation graph stored in Apache AGE (graph name: case_graph). "
    "Edges are (a)-[:REF]->(b) meaning 'a cites b'. "
    "Given seed case ids (from keyword_case_search and semantic_case_search), "
    "returns a ranked set of related precedent cases via multi-hop traversal, "
    "enriched with metadata from the relational cases table."
))
def precedent_graph_search(
    seed_case_ids: Annotated[List[int], Field(
        description=(
            "List of seed case ids from Tool 1 and/or Tool 2. "
            "Pass the union of ids from both tools for best coverage."
        )
    )],
    direction: Annotated[Literal["cites", "cited_by", "both"], Field(
        default="both",
        description=(
            "'cites' = follow outgoing REF edges (seed cites prior authority). "
            "'cited_by' = follow incoming REF edges (later cases cite the seed). "
            "'both' = union of both directions."
        )
    )] = "both",
    hops: Annotated[int, Field(
        default=2,
        description="Max traversal depth. 1 = direct citations only. 2 = cite-of-cite expansion. Max 3."
    )] = 2,
    prefer_supreme: Annotated[bool, Field(
        default=True,
        description="If true, weight Washington Supreme Court higher than Court of Appeals when ranking."
    )] = True,
) -> str:
    print("precedent_graph_search was called")
    print(f"  seeds={len(seed_case_ids)}  direction={direction}  hops={hops}  limit=5  prefer_supreme={prefer_supreme}")

    if not seed_case_ids:
        return "No seed_case_ids provided."

    hops = max(1, min(hops, 3))

    # Graph stores case_id as text property, created from temp_cases.data->>'id'
    # Your helper functions store it as text, so we compare using string literals.
    seed_literals = ", ".join([f"'{int(i)}'" for i in sorted(set(seed_case_ids))])

    # Build Cypher fragments for each direction.
    # REF direction: (a)-[:REF]->(b) means "a cites b"
    # - cites: seed -> authority
    # - cited_by: later_case -> seed  (we traverse reversed by matching (n)-[:REF*]->(seed))
    cypher_queries = []

    if direction in ("cites", "both"):
        cypher_queries.append(f"""
            SELECT * FROM cypher('case_graph', $$
                MATCH (s:case)-[:REF*1..{hops}]->(t:case)
                WHERE s.case_id IN [{seed_literals}]
                RETURN t.case_id AS related_id, count(*) AS paths
            $$) AS (related_id agtype, paths agtype);
        """)

    if direction in ("cited_by", "both"):
        cypher_queries.append(f"""
            SELECT * FROM cypher('case_graph', $$
                MATCH (t:case)-[:REF*1..{hops}]->(s:case)
                WHERE s.case_id IN [{seed_literals}]
                RETURN t.case_id AS related_id, count(*) AS paths
            $$) AS (related_id agtype, paths agtype);
        """)

    # Execute graph traversal and aggregate path counts.
    related_paths = {}  # related_id(int) -> paths(int)
    try:
        with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
            # AGE session setup            
            cur.execute("SET search_path = public, ag_catalog, \"$user\";")

            for q in cypher_queries:
                cur.execute(q)
                for related_id_ag, paths_ag in cur.fetchall():
                    # AGE returns agtype; psycopg gives it as Python str like '"123"' or '123'
                    rid_str = str(related_id_ag).strip('"')
                    if not rid_str.isdigit():
                        continue
                    rid = int(rid_str)

                    # Skip seeds in the related set; we only want expansions
                    if rid in seed_case_ids:
                        continue

                    p = int(str(paths_ag).strip('"')) if str(paths_ag).strip('"').isdigit() else 1
                    related_paths[rid] = related_paths.get(rid, 0) + p

    except Exception as e:
        return f"Graph query failed. Is case_graph created and populated? Error: {e}"

    if not related_paths:
        return "No related cases found from the citation graph for the provided seeds."

    related_ids = list(related_paths.keys())

    # Enrich from relational table so Tool 4 can immediately extract from opinion text.
    # Also compute a simple ranking score: paths (centrality proxy) + court weighting.
    try:
        with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
            cur.execute(
                """
                SELECT id, name, decision_date, court_level, opinion
                FROM public.cases
                WHERE id = ANY(%s);
                """,
                (related_ids,)
            )
            rows = cur.fetchall()
    except Exception as e:
        return f"Failed to fetch case metadata from relational table. Error: {e}"

    # Court weighting: Supreme gets a boost if prefer_supreme=True
    def court_weight(level: str) -> int:
        if not prefer_supreme:
            return 0
        if level == "Washington Supreme Court":
            return 2
        if level == "Washington Court of Appeals":
            return 0
        return 0

    # Build ranked output
    ranked = []
    for case_id, name, decision_date, court_level, opinion in rows:
        paths = related_paths.get(case_id, 0)
        score = paths + court_weight(court_level)
        snippet = (opinion or "").replace("\n", " ")[:240]
        ranked.append((score, paths, case_id, decision_date, court_level, name, snippet))

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    print(f"  related_count={len(ranked)}")

    # Return a parseable, stable text format:
    # - first section: ids only (for Tool 4 handoff)
    # - second section: human-readable justification
    top_ids = [str(r[2]) for r in ranked]
    lines = []
    lines.append("RELATED_CASE_IDS: " + ", ".join(top_ids))
    lines.append("")
    lines.append("Related cases (ranked):")
    for score, paths, case_id, decision_date, court_level, name, snippet in ranked:
        lines.append(
            f"{case_id} | {decision_date} | {court_level} | paths={paths} | score={score} | {name}: {snippet}..."
        )

    return "\n".join(lines)


## Test Tool 3 Directly

In [ ]:
def parse_ids(tool_output: str) -> list[int]:
    ids = []
    for line in tool_output.splitlines():
        m = re.match(r"^(\d+)\s+\|", line)
        if m:
            ids.append(int(m.group(1)))
    return ids

# Example: run Tool 1 and Tool 2 quickly (or reuse saved strings)
kw_out = keyword_case_search(
    "'common enemy'~3 AND (surface water OR drainage OR stormwater)",
    court_level="Washington Supreme Court",
    min_year=1900,
)

sem_out = semantic_case_search(
    "developer regraded lot causing flooding; city changed road drainage",
    min_year=1900,
    use_advanced_filtering=True,
)

seed_ids = sorted(set(parse_ids(kw_out) + parse_ids(sem_out)))
print("Seed ids:", seed_ids)

print("")
print("// Tool 3: Graph traversal //")
graph_out = precedent_graph_search(
    seed_case_ids=seed_ids,
    direction="both",
    hops=2,
    prefer_supreme=True,
)
print(graph_out)

## Re-assmble Agent and integrate Tool 3

In [ ]:
client2 = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

agent2 = client2.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, why each is relevant, "
        "and a short one-sentence summary of the opinion. Run each tool only once."
    ),
    tools=[keyword_case_search,semantic_case_search,precedent_graph_search]
)

user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the road."
)

print("// Functions the Agent Called: //")
result = await agent2.run(user_query)

print("")
print("// Agent Response: //")
Markdown(result.text)

## Tool 4 - Extract() entitied from results of found cases

In [ ]:
@tool(description=(
    "Extract structured legal facts from one or more case opinions using azure_ai.extract. "
    "Returns JSON per case with compact fields (holding and issues by default). "
    "Use after the graph tool to turn raw opinions into brief-ready structured records."
))
def case_analyst_extract(
    case_ids: Annotated[List[int], Field(
        description=(
            "List of case ids to analyze (typically from precedent_graph_search's RELATED_CASE_IDS)."
        )
    )],
    model_alias: Annotated[str, Field(
        default="default-chat",
        description="azure_ai model alias. Default 'default-chat' uses managed models."
    )] = "default-chat",
    include_snippet: Annotated[bool, Field(
        default=True,
        description="Include a short opinion snippet for debugging."
    )] = True,
    snippet_chars: Annotated[int, Field(
        default=200,
        description="Max characters of snippet."
    )] = 200,
    max_opinion_chars: Annotated[int, Field(
        default=1200,
        description="Max number of characters from opinion sent to the model."
    )] = 1200,
) -> str:
    print("case_analyst_extract was called")
    print(f"  case_ids={len(case_ids)}  model_alias={model_alias}")

    if not case_ids:
        return "No case_ids provided."

    # Fields to Extract (always use the default set)
    fields = ["holding", "issues", "statutes_cited", "disposition"]

    # Emit the entity fields the agent is extracting (parsed by the UI trace panel)
    print(f"  fields={','.join(fields)}")

    snippet_chars = max(0, min(snippet_chars, 500))
    max_opinion_chars = max(500, min(max_opinion_chars, 4000))

    # Step 1: Fetch case rows in a single query
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS azure_ai;")
        cur.execute(
            """
            SELECT id, name, decision_date, court_level, opinion
            FROM public.cases
            WHERE id = ANY(%s);
            """,
            (case_ids,),
        )
        rows = cur.fetchall()

    if not rows:
        return "No rows found for the provided case_ids."

    # Step 2: Extract in parallel — each thread opens its own connection
    # so azure_ai.extract() LLM calls run concurrently instead of sequentially.
    def _extract_one(row):
        case_id, name, decision_date, court_level, opinion = row
        short_opinion = (opinion or "")[:max_opinion_chars]
        try:
            with psycopg.connect(**DB_CONFIG, autocommit=True) as c2, c2.cursor() as cur2:
                cur2.execute(
                    "SELECT azure_ai.extract(%s, %s, %s);",
                    (short_opinion, fields, model_alias),
                )
                extracted = cur2.fetchone()[0]
        except Exception as e:
            extracted = {"error": "extraction_failed", "reason": str(e)}

        card = {
            "id": case_id,
            "name": name,
            "decision_date": str(decision_date),
            "court_level": court_level,
            "extracted": extracted,
        }
        if include_snippet:
            card["snippet"] = (opinion or "").replace("\n", " ")[:snippet_chars]
        return card

    with ThreadPoolExecutor(max_workers=len(rows)) as pool:
        futures = {pool.submit(_extract_one, r): r[0] for r in rows}
        results = []
        for fut in as_completed(futures):
            card = fut.result()
            results.append(card)
            # Emit per-case completion line for the UI trace panel
            cname = (card.get("name") or "").replace("|", "/")[:60]
            print(f"  extracted_case={card.get('id')}|{cname}", flush=True)

    # Preserve original order (by case_id position in the input list)
    id_order = {cid: i for i, cid in enumerate(case_ids)}
    results.sort(key=lambda c: id_order.get(c["id"], 1_000_000))

    return json.dumps(results, indent=2, default=str)


## Test Tool 4

In [ ]:
def parse_related_case_ids(graph_output: str) -> list[int]:
    m = re.search(r"^RELATED_CASE_IDS:\s*(.+)$", graph_output, flags=re.MULTILINE)
    if not m:
        return []
    raw = m.group(1)
    ids = []
    for part in raw.split(","):
        part = part.strip()
        if part.isdigit():
            ids.append(int(part))
    return ids


def parse_ids(tool_output: str) -> list[int]:
    ids = []
    for line in tool_output.splitlines():
        m = re.match(r"^(\d+)\s+\|", line)
        if m:
            ids.append(int(m.group(1)))
    return ids

# Example: run Tool 1 and Tool 2 quickly (or reuse saved strings)
kw_out = keyword_case_search(
    "'common enemy'~3 AND (surface water OR drainage OR stormwater)",
    court_level="Washington Supreme Court",
    min_year=1900,
)

sem_out = semantic_case_search(
    "developer regraded lot causing flooding; city changed road drainage",
    min_year=1900,
    use_advanced_filtering=True,
)

seed_ids = sorted(set(parse_ids(kw_out) + parse_ids(sem_out)))
print("Seed ids:", seed_ids)

print("")
print("// Tool 3: Graph traversal //")
graph_out = precedent_graph_search(
    seed_case_ids=seed_ids,
    direction="both",
    hops=2,
    prefer_supreme=True,
)
print(graph_out)

print("// Tool 4: Case analyst extraction (from Tool 3 output) //")

case_ids = parse_related_case_ids(graph_out)
print("Parsed case_ids:", case_ids[:10], "..." if len(case_ids) > 10 else "")

analysis_json = case_analyst_extract(
    case_ids=case_ids,
    model_alias="default-chat",
    include_snippet=True,
)

print("")
print("// Tool 4 Output (formatted) //")

if analysis_json.startswith("CASE_BRIEFS_JSON:"):
    json_str = analysis_json.replace("CASE_BRIEFS_JSON: ", "")
    data = json.loads(json_str)

    # Pretty JSON (full structure)
    print("\n=== Full JSON (pretty) ===")
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    print("Unexpected output format:")
    print(analysis_json[:1000])

## Re-assemble with Tool 4 and test run Agent

In [ ]:
client2 = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

agent2 = client2.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with the case names, why each is relevant, "
        "and a short one-sentence summary of the opinion. Run each tool only once."
    ),
    tools=[keyword_case_search,semantic_case_search,precedent_graph_search,case_analyst_extract]
)

user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the developer "
    "next door regraded their lot and the city redid the drainage on the road."
)

print("// Functions the Agent Called: //")
result = await agent2.run(user_query)

print("")
print("// Agent Response: //")
Markdown(result.text)

## Tool 5

In [ ]:
@tool(description=(
    "Retrieve historical precipitation data for a location and date range to support "
    "weather-related legal claims (flooding, storm damage, drainage disputes). "
    "Uses the Open-Meteo Archive API. This tool is OPTIONAL — only call it when the "
    "legal question involves weather, water, storms, flooding, drainage, or precipitation."
))
def get_weather_evidence(
    latitude: Annotated[float, Field(
        description="Latitude of the location (WGS84). Example: 47.6062 for Seattle."
    )],
    longitude: Annotated[float, Field(
        description="Longitude of the location (WGS84). Example: -122.3321 for Seattle."
    )],
    start_date: Annotated[str, Field(
        description="Start date in YYYY-MM-DD format."
    )],
    end_date: Annotated[Optional[str], Field(
        default=None,
        description="End date in YYYY-MM-DD format. Defaults to start_date (single day)."
    )] = None,
) -> str:
    print("get_weather_evidence was called")
    print(f"  lat={latitude}  lon={longitude}  start={start_date}  end={end_date or start_date}")

    if end_date is None:
        end_date = start_date

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "precipitation_sum",
        "timezone": "UTC",
    }

    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f"WEATHER_EVIDENCE_ERROR: Failed to retrieve weather data: {e}"

    dates = data.get("daily", {}).get("time", [])
    precip = data.get("daily", {}).get("precipitation_sum", [])

    if not dates:
        return "WEATHER_EVIDENCE: No data returned for the given parameters."

    records = []
    total_mm = 0.0
    for d, p in zip(dates, precip):
        rain = p if p is not None else 0.0
        total_mm += rain
        records.append({"date": d, "precipitation_mm": rain})

    summary = {
        "location": {"latitude": latitude, "longitude": longitude},
        "period": {"start": start_date, "end": end_date},
        "total_precipitation_mm": round(total_mm, 2),
        "days_with_rain": sum(1 for r in records if r["precipitation_mm"] > 0),
        "total_days": len(records),
        "daily_records": records if len(records) <= 14 else records[:7] + records[-7:],
    }

    return "WEATHER_EVIDENCE_JSON: " + json.dumps(summary, ensure_ascii=False)

### Test Tool 5 flow directly

In [ ]:
def parse_related_case_ids(graph_output: str) -> list[int]:
    m = re.search(r"^RELATED_CASE_IDS:\s*(.+)$", graph_output, flags=re.MULTILINE)
    if not m:
        return []
    raw = m.group(1)
    ids = []
    for part in raw.split(","):
        part = part.strip()
        if part.isdigit():
            ids.append(int(part))
    return ids


def parse_ids(tool_output: str) -> list[int]:
    ids = []
    for line in tool_output.splitlines():
        m = re.match(r"^(\d+)\s+\|", line)
        if m:
            ids.append(int(m.group(1)))
    return ids

# Example: run Tool 1 and Tool 2 quickly (or reuse saved strings)
kw_out = keyword_case_search(
    "'common enemy'~3 AND (surface water OR drainage OR stormwater)",
    court_level="Washington Supreme Court",
    min_year=1900,
)

sem_out = semantic_case_search(
    "developer regraded lot causing flooding; city changed road drainage",
    min_year=1900,
    use_advanced_filtering=True,
)

seed_ids = sorted(set(parse_ids(kw_out) + parse_ids(sem_out)))
print("Seed ids:", seed_ids)

print("")
print("// Tool 3: Graph traversal //")
graph_out = precedent_graph_search(
    seed_case_ids=seed_ids,
    direction="both",
    hops=2,
    prefer_supreme=True,
)
print(graph_out)

print("// Tool 4: Case analyst extraction (from Tool 3 output) //")

case_ids = parse_related_case_ids(graph_out)
print("Parsed case_ids:", case_ids[:10], "..." if len(case_ids) > 10 else "")

analysis_json = case_analyst_extract(
    case_ids=case_ids,
    model_alias="default-chat",
    include_snippet=True,
)

print("")
print("// Tool 4 Output (formatted) //")

if analysis_json.startswith("CASE_BRIEFS_JSON:"):
    json_str = analysis_json.replace("CASE_BRIEFS_JSON: ", "")
    data = json.loads(json_str)

    # Pretty JSON (full structure)
    print("\n=== Full JSON (pretty) ===")
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    print("Unexpected output format:")
    print(analysis_json[:1000])



print("")
print("// Tool 5: Weather Evidence (optional – weather-related query) //")

weather_out = get_weather_evidence(
    latitude=47.6062,
    longitude=-122.3321,
    start_date="2023-01-10",
    end_date="2023-01-20",
)

if weather_out.startswith("WEATHER_EVIDENCE_JSON:"):
    wdata = json.loads(weather_out.replace("WEATHER_EVIDENCE_JSON: ", ""))
    print("\n=== Weather Evidence (pretty) ===")
    print(json.dumps(wdata, indent=2, ensure_ascii=False))
else:
    print(weather_out)


### Re-assemble Agent and test all 5 tools together.

In [ ]:
client2 = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

agent2 = client2.as_agent(
    instructions=(
        "You are Counsel, an AI legal associate helping analyze Washington State legal cases.\n\n"

        "You have access to a multi-step legal analysis pipeline.\n\n"

        "Required Workflow (always run these 4 tools in order):\n"
        "1. Use keyword_case_search to identify the governing legal doctrine and canonical cases.\n"
        "2. Use semantic_case_search to find factually similar cases using vector similarity.\n"
        "3. Combine results and use precedent_graph_search to identify the strongest line of authority.\n"
        "4. Use case_analyst_extract to extract structured legal facts from the most relevant cases.\n\n"

        "Weather / Storm / Flooding Tool:\n"
        "5. If the question involves weather, flooding, storms, precipitation, drainage, or water damage,\n"
        "   use get_weather_evidence to retrieve historical precipitation data for the relevant location and date.\n"
        "   Include this data as supporting evidence in your analysis.\n\n"

        "Important constraints:\n"
        "- DO NOT skip the required steps (1-4).\n"
        "- DO NOT manually summarize cases without using tools.\n"
        "- Always pass outputs between tools (e.g., case ids, JSON).\n"
        "- Call each tool AT MOST ONCE. Never call the same tool twice.\n\n"

        "Final Output:\n"
        "After all tool calls complete, synthesize the results into a comprehensive legal analysis.\n"
        "Use IRAC format (Issue, Rule, Application, Conclusion).\n"
        "Cite case names from the extracted data. If weather evidence was gathered, incorporate it.\n"
        "Do NOT include raw tool outputs — present a polished analysis.\n\n"

        "Your job is to orchestrate the tools, then produce the final analysis yourself."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search,
        case_analyst_extract,
        get_weather_evidence
    ]
)

user_query = (
    "I have a client in Seattle, Washington, whose property floods after a neighboring developer "
    "regraded their lot and the city modified road drainage. Analyze potential legal liability "
    "and produce a structured legal analysis and draft brief."
)

print("// Functions the Agent Called: //")
result = await agent2.run(user_query)

print("")
print("// Agent Response: //")
Markdown(result.text)

### Integrate Agent into an app UI with all 5 Tools running together

In [ ]:
# ── Create the Counsel Agent for the UI ──────────────────────────────

_ui_client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

_ui_agent = _ui_client.as_agent(
    instructions=(
        "You are Counsel, an AI legal research assistant specializing in Washington State case law.\n\n"

        "You have access to a multi-step legal analysis pipeline.\n\n"

        "Required Workflow (always run these 4 tools in order):\n"
        "1. keyword_case_search – BM25 full-text search for canonical cases\n"
        "2. semantic_case_search – DiskANN vector search for factually similar cases\n"
        "3. precedent_graph_search – Citation graph traversal via Apache AGE\n"
        "4. case_analyst_extract – Extract structured legal facts via azure_ai.extract\n\n"

        "Weather / Storm / Flooding Tool:\n"
        "5. Always run this tool last if ran. If the question involves weather, flooding, storms, precipitation, drainage, or water damage,\n"
        "   use get_weather_evidence to retrieve historical precipitation data for the relevant location and date.\n"
        "   Include this data as supporting evidence in your analysis.\n\n"

        "Constraints:\n"
        "- Do NOT skip the required steps (1-4).\n"
        "- If any mention of flooding, storms, drainage, or water damage is detected, you must use the Weather / Storm / Flooding Tool. And run this tool last.\n"
        "- Pass outputs between tools (case ids, JSON payloads).\n"
        "- Do NOT manually summarize without tools.\n"
        "- Call each tool AT MOST ONCE. Never call the same tool twice.\n\n"

        "Final Response Format (Markdown):\n"
        "After all tool calls complete, synthesize the results into a polished legal analysis.\n"
        "- A short opening paragraph summarizing the key legal holding and its significance.\n"
        "- A short paragraph distinguishing the most important prior cases.\n"
        "- If weather evidence was gathered, a paragraph incorporating precipitation data as supporting evidence.\n"
        "- **Key Takeaway:** A concise statement of the principal legal rule.\n"
        "- **Key Cases in this Line of Authority:** A bulleted list of relevant cases, each with case name, citation, and year.\n"
        "- **Statutes Extracted:** A bulleted list of statutes and what case it is from.\n"
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search,
        case_analyst_extract,
        get_weather_evidence,
    ],
)

# ── Tool Trace Metadata ──────────────────────────────────────────────
_TOOL_META = {
    "keyword_case_search":    {"idx": 1, "name": "KeywordCaseSearchTool",  "tech": "pg_fts BM25"},
    "semantic_case_search":   {"idx": 2, "name": "SemanticCaseSearchTool", "tech": "DiskANN"},
    "precedent_graph_search": {"idx": 3, "name": "PrecedentGraphTool",     "tech": "AGE Graph"},
    "case_analyst_extract":   {"idx": 4, "name": "CaseAnalystTool",        "tech": "azure_ai.extract"},
    "get_weather_evidence":   {"idx": 5, "name": "WeatherEvidenceTool",    "tech": "Open-Meteo API"},
}

_TRACE_PALETTE = {
    "accent": "#5b5ef7",
    "accent_soft": "#eef0ff",
    "accent_border": "#d8ddff",
    "text": "#1f2340",
    "muted": "#687087",
    "panel": "#ffffff",
    "line": "#dfe3f0",
    "success": "#217346",
    "success_soft": "#edf8f0",
    "success_border": "#cce8d5",
    "success_text": "#1f5f3c",
    "shadow": "0 10px 28px rgba(31, 35, 64, 0.08)",
}

_UI_THEME = gr.themes.Soft(
    primary_hue="indigo",
    neutral_hue="gray",
    radius_size=gr.themes.sizes.radius_lg,
)


def _tool_icon_svg(key: str) -> str:
    accent = _TRACE_PALETTE["accent"]
    icons = {
        "keyword_case_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<circle cx="8.5" cy="8.5" r="5.5" stroke="{accent}" stroke-width="1.8"/>'
            f'<path d="M12.5 12.5L16.5 16.5" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "semantic_case_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<ellipse cx="10" cy="5" rx="5.8" ry="2.6" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M4.2 5V10.2C4.2 11.6 6.8 12.8 10 12.8C13.2 12.8 15.8 11.6 15.8 10.2V5" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M4.2 10.2V15C4.2 16.4 6.8 17.6 10 17.6C13.2 17.6 15.8 16.4 15.8 15V10.2" stroke="{accent}" stroke-width="1.6"/>'
            f'</svg>'
        ),
        "precedent_graph_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<circle cx="5" cy="10" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<circle cx="15" cy="5" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<circle cx="15" cy="15" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M6.5 9L13.2 5.8" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M6.5 11L13.2 14.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M15 6.8V13.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "case_analyst_extract": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<path d="M6 2.8H11.8L15.5 6.5V16.5C15.5 17.3 14.8 18 14 18H6C5.2 18 4.5 17.3 4.5 16.5V4.3C4.5 3.5 5.2 2.8 6 2.8Z" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M11.5 2.8V6.4H15.2" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M7.5 10H12.5" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M7.5 13H11.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "get_weather_evidence": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<path d="M6 14C3.8 14 2 12.4 2 10.4C2 8.7 3.3 7.3 5 7C5.5 4.7 7.5 3 10 3C12.8 3 15 5 15.3 7.5C17 7.8 18 9.2 18 10.8C18 12.6 16.5 14 14.7 14H6Z" stroke="{accent}" stroke-width="1.6" stroke-linejoin="round"/>'
            f'<path d="M8 16V14" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M12 17V14" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M10 18V15" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
    }
    return icons.get(key, "")


def _header_icon_svg() -> str:
    accent = _TRACE_PALETTE["accent"]
    return (
        f'<svg width="28" height="28" viewBox="0 0 24 24" fill="none" '
        f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
        f'<path d="M4 7H20" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'<path d="M12 5V19" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'<path d="M7 7C7 10 5.5 12 4 13.3C5 14.1 6.2 14.5 7.5 14.5C9.4 14.5 10.9 13.4 12 11.8" stroke="{accent}" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>'
        f'<path d="M17 7C17 10 18.5 12 20 13.3C19 14.1 17.8 14.5 16.5 14.5C14.6 14.5 13.1 13.4 12 11.8" stroke="{accent}" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>'
        f'<path d="M9 19H15" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'</svg>'
    )


# ── Trace Parsing Helpers ────────────────────────────────────────────

def _parse_traces(captured: str) -> list[dict]:
    """Parse captured stdout to identify which tools the agent called."""
    traces, lines = [], captured.split("\n")
    for i, line in enumerate(lines):
        for key, meta in _TOOL_META.items():
            if f"{key} was called" in line:
                details = []
                for nxt in lines[i + 1:]:
                    if nxt.startswith("  "):
                        details.append(nxt.strip())
                    else:
                        break
                traces.append({"key": key, "details": details, **meta})
                break
    return traces


def _new_trace_state() -> dict:
    return {
        "traces": [],
        "line_buffer": "",
        "changed": False,
    }


def _ingest_trace_chunk(state: dict, chunk: str, now: float | None = None) -> None:
    """Update trace state as stdout arrives so the UI stays in sync with actual tool starts."""
    if not chunk:
        return

    if now is None:
        now = time.perf_counter()

    state["line_buffer"] += chunk
    while "\n" in state["line_buffer"]:
        line, state["line_buffer"] = state["line_buffer"].split("\n", 1)
        stripped = line.rstrip("\r")

        matched_key = None
        for key in _TOOL_META:
            if f"{key} was called" in stripped:
                matched_key = key
                break

        if matched_key is not None:
            trace = {"key": matched_key, "details": [], "started_at": now, **_TOOL_META[matched_key]}
            state["traces"].append(trace)
            state["changed"] = True
            continue

        if stripped.startswith("  ") and state["traces"]:
            state["traces"][-1]["details"].append(stripped.strip())
            state["changed"] = True


def _run_ui_agent(message: str):
    """Run the async agent on a worker thread so UI updates are not blocked by sync tool calls."""
    return asyncio.run(_ui_agent.run(message))


def _trace_desc(t: dict) -> str:
    d = " ".join(t.get("details", []))
    k = t["key"]
    if k == "keyword_case_search":
        m = re.search(r"query=(.+?)\s\s+court_level=", d)
        if m:
            q = m.group(1).strip().strip("'\"").replace("\\'", "'").replace('\\"', '"')
        else:
            q = "case law terms"
        q = q[:55]
        return f'Searched for "{q}" and related terms.'
    if k == "semantic_case_search":
        parts = []
        m = re.search(r"court_level=([^\s]+)", d)
        if m and m.group(1) != "None":
            parts.append(f"court_level IN ({m.group(1)})")
        m = re.search(r"min_year=(\d+)", d)
        if m:
            parts.append(f"decision_date >= {m.group(1)}")
        filt = ", ".join(parts) if parts else "no additional filters"
        return f"Expanded search using vector similarity with filters: {filt}."
    if k == "precedent_graph_search":
        m = re.search(r"seeds=(\d+)", d)
        n = m.group(1) if m else "N"
        return f"{n} unique cases found via KeywordCaseSearchTool and SemanticCaseSearchTool."
    if k == "case_analyst_extract":
        fm = re.search(r"fields=([^\s]+)", d)
        cm = re.search(r"case_ids=(\d+)", d)
        n_in = cm.group(1) if cm else "the"
        if fm:
            field_list = [f.strip() for f in fm.group(1).split(",") if f.strip()]
            chips = "".join(
                f'<span style="display:inline-block;padding:2px 8px;margin:2px 4px 2px 0;'
                f'background:#eef0ff;border:1px solid #d8ddff;border-radius:999px;'
                f'font-size:11px;color:#4f56c7;">{f}</span>'
                for f in field_list
            )
            return (
                f'Extracting structured legal entities from {n_in} case opinions:'
                f'<div style="margin-top:6px;">{chips}</div>'
            )
        return f"Extracting structured fields from {n_in} key opinions."
    if k == "get_weather_evidence":
        m = re.search(r"start=(\S+)", d)
        dt = m.group(1) if m else "requested dates"
        return f"Retrieved historical precipitation data for {dt}."
    return ""


def _trace_result(t: dict) -> str:
    d = " ".join(t.get("details", []))
    k = t["key"]
    if k == "keyword_case_search":
        return "Results: 5 cases"
    if k == "semantic_case_search":
        return "Results: 5 cases"
    if k == "precedent_graph_search":
        rm = re.search(r"related_count=(\d+)", d)
        hm = re.search(r"hops=(\d+)", d)
        if rm:
            n = rm.group(1)
            hops = hm.group(1) if hm else "2"
            return f"{n} additional related cases found traversing the graph using {hops} hops"
        return "Paths found via graph traversal"
    if k == "case_analyst_extract":
        cases = re.findall(r"extracted_case=(\d+)\|(.+?)(?=\s+extracted_case=|\s+\w+=|$)", d)
        if cases:
            items = "".join(
                f'<div style="display:flex;align-items:flex-start;gap:6px;margin-top:4px;">'
                f'<span style="color:#217346;font-weight:700;">✓</span>'
                f'<span style="font-size:12px;color:#1f2340;">{name.strip()} '
                f'<span style="color:#687087;">(id {cid})</span></span>'
                f'</div>'
                for cid, name in cases
            )
            return (
                f'<div style="font-weight:600;color:#1f5f3c;margin-top:4px;">'
                f'Extracted {len(cases)} case{"s" if len(cases) != 1 else ""}:</div>'
                f'{items}'
            )
        m = re.search(r"case_ids=(\d+)", d)
        return f"Cases analyzed: {m.group(1) if m else 'N'}"
    return ""


# ── Trace Panel HTML Builder (Light Theme + Flat Icons) ─────────────

def _build_trace_html(traces, t0, running=False, total_time=None):
    """Build trace panel HTML. Supports live updates during streaming."""
    panel = _TRACE_PALETTE["panel"]
    text = _TRACE_PALETTE["text"]
    muted = _TRACE_PALETTE["muted"]
    accent = _TRACE_PALETTE["accent"]
    accent_soft = _TRACE_PALETTE["accent_soft"]
    accent_border = _TRACE_PALETTE["accent_border"]
    line = _TRACE_PALETTE["line"]
    success = _TRACE_PALETTE["success"]
    success_soft = _TRACE_PALETTE["success_soft"]
    success_border = _TRACE_PALETTE["success_border"]
    success_text = _TRACE_PALETTE["success_text"]
    shadow = _TRACE_PALETTE["shadow"]
    now = time.perf_counter()

    if not traces and not running:
        return (
            f'<div style="padding:24px 22px;text-align:center;color:{muted};background:{panel};">'
            f'<div style="width:52px;height:52px;border-radius:16px;background:{accent_soft};margin:0 auto 12px;'
            f'display:flex;align-items:center;justify-content:center;color:{accent};font-size:22px;">⌕</div>'
            f'<div style="font-weight:600;color:{text};margin-bottom:4px;">Tool Trace</div>'
            f'<div>Ask a question to see the research steps</div>'
            f'</div>'
        )
    if not traces and running:
        return (
            f'<div style="padding:24px 22px;text-align:center;color:{muted};background:{panel};">'
            f'<div style="width:52px;height:52px;border-radius:16px;background:{accent_soft};margin:0 auto 12px;'
            f'display:flex;align-items:center;justify-content:center;color:{accent};font-size:22px;" class="spin">◌</div>'
            f'<div style="font-weight:600;color:{text};margin-bottom:4px;">Starting analysis pipeline</div>'
            f'<div>Preparing the first tool call</div>'
            f'</div>'
        )

    n_label = f'{len(traces)} tool{"s" if len(traces) != 1 else ""}'
    state = "running" if running else "used"
    html = (
        f'<div style="background:{panel};padding:2px 2px 8px 2px;">'
        f'<div style="font-weight:700;font-size:15px;color:{text};margin-bottom:14px;">'
        f'Tool Trace <span style="font-weight:500;color:{muted};">({n_label} {state})</span></div>'
    )

    for i, t in enumerate(traces):
        is_last = i == len(traces) - 1
        is_active = running and is_last

        started_at = t.get("started_at", t0)
        if is_active:
            dur = now - started_at
        elif i + 1 < len(traces):
            dur = traces[i + 1].get("started_at", started_at) - started_at
        else:
            end_at = t0 + (total_time or 0)
            dur = end_at - started_at
        dur = max(dur, 0)

        ts = f"{int(dur * 1000)} ms" if dur < 1 else f"{dur:.1f}s"
        desc = _trace_desc(t)
        res = _trace_result(t)
        icon = _tool_icon_svg(t["key"])

        if is_active:
            status_icon = f'<span style="color:{accent};font-size:14px;" class="spin">◌</span>'
            row_bg = "#f8f9ff"
        else:
            status_icon = f'<span style="color:{success};font-size:14px;">●</span>'
            row_bg = panel

        connector = ""
        if i < len(traces) - 1:
            connector = (
                f'<div style="position:absolute;left:15px;top:34px;width:2px;height:40px;'
                f'background:{accent_border};border-radius:999px;"></div>'
            )

        res_div = (
            f'<div style="color:{muted};font-size:13px;margin-top:6px;">{res}</div>'
            if res and (not is_active or t["key"] == "case_analyst_extract") else ""
        )

        html += (
            f'<div style="display:flex;gap:12px;position:relative;margin-bottom:14px;">'
            f'<div style="position:relative;width:32px;flex:0 0 32px;display:flex;justify-content:center;">'
            f'<div style="width:30px;height:30px;border-radius:999px;background:{accent};color:#fff;' 
            f'display:flex;align-items:center;justify-content:center;font-weight:700;font-size:13px;">{t["idx"]}</div>'
            f'{connector}</div>'
            f'<div style="flex:1;background:{row_bg};border:1px solid {line};border-radius:14px;padding:12px 14px;'
            f'box-shadow:{shadow};">'
            f'<div style="display:flex;justify-content:space-between;align-items:flex-start;gap:12px;">'
            f'<div style="display:flex;gap:10px;align-items:flex-start;min-width:0;">'
            f'<div style="width:20px;height:20px;display:flex;align-items:center;justify-content:center;flex:0 0 20px;">{icon}</div>'
            f'<div style="min-width:0;">'
            f'<div style="font-weight:700;color:{text};line-height:1.25;">{t["name"]} '
            f'<span style="font-weight:500;color:{muted};font-size:12px;">({t["tech"]})</span></div>'
            f'<div style="color:{muted};font-size:13px;line-height:1.45;margin-top:4px;">{desc}</div>'
            f'{res_div}</div></div>'
            f'<div style="text-align:right;white-space:nowrap;color:{muted};font-size:13px;">{ts}<div style="margin-top:4px;">{status_icon}</div></div>'
            f'</div></div></div>'
        )

    if running:
        elapsed = now - t0
        html += (
            f'<div style="display:flex;align-items:center;gap:8px;padding:12px 14px;border-radius:12px;' 
            f'background:{accent_soft};border:1px solid {accent_border};">'
            f'<span class="spin" style="color:{text};">◌</span>'
            f'<span style="font-weight:700;color:{text};">Running... {elapsed:.1f}s elapsed</span></div>'
        )
    else:
        t_disp = total_time or 0
        html += (
            f'<div style="display:flex;align-items:center;gap:8px;padding:12px 14px;border-radius:12px;' 
            f'background:{success_soft};border:1px solid {success_border};color:{success_text};">'
            f'<span style="font-size:14px;color:{success};"></span>'
            f'<span style="font-weight:700;color:{success_text};">Completed in {t_disp:.2f} seconds</span></div>'
        )

    html += "</div>"
    return html


# ── Chat Handler (async generator for live streaming) ────────────────

async def _chat_fn(message: str, history: list):
    if not message or not message.strip():
        yield history, _build_trace_html([], 0), ""
        return

    old_stdout = sys.stdout
    buf = io.StringIO()
    trace_state = _new_trace_state()

    class _Tee:
        def write(self, s):
            buf.write(s)
            _ingest_trace_chunk(trace_state, s)
            old_stdout.write(s)
        def flush(self):
            buf.flush()
            old_stdout.flush()
        def __getattr__(self, name):
            return getattr(old_stdout, name)
    sys.stdout = _Tee()

    t0 = time.perf_counter()
    pending_history = history + [{"role": "user", "content": message}]
    yield pending_history, _build_trace_html([], t0, running=True), ""

    task = asyncio.create_task(asyncio.to_thread(_run_ui_agent, message))
    last_render = 0.0

    while not task.done():
        await asyncio.sleep(0.15)
        now = time.perf_counter()
        should_render = trace_state["changed"] or trace_state["traces"] or (now - last_render >= 0.5)
        if not should_render:
            continue

        trace_state["changed"] = False
        last_render = now
        trace_html = _build_trace_html(trace_state["traces"], t0, running=True)
        yield pending_history, trace_html, ""

    elapsed = time.perf_counter() - t0
    sys.stdout = old_stdout

    try:
        res = task.result()
        response = res.text
    except Exception as e:
        response = f"⚠️ An error occurred: {e}"

    parsed = _parse_traces(buf.getvalue())
    if len(parsed) > len(trace_state["traces"]):
        for i, parsed_trace in enumerate(parsed):
            if i < len(trace_state["traces"]):
                continue
            parsed_trace["started_at"] = t0 + elapsed
            trace_state["traces"].append(parsed_trace)

    trace_html = _build_trace_html(trace_state["traces"], t0, running=False, total_time=elapsed)

    final_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response},
    ]
    yield final_history, trace_html, ""


# ── Status Bar ───────────────────────────────────────────────────────

def _status_html() -> str:
    now = datetime.now().strftime("%I:%M %p")
    text = _TRACE_PALETTE["muted"]
    success = _TRACE_PALETTE["success"]
    return (
        f'<div style="display:flex;align-items:center;gap:6px;padding:8px 16px;'
        f'font-size:12px;color:{text};flex-wrap:wrap;">'
        f'Connected to HorizonDB (Reader Endpoint) '
        f'<span style="color:{success};">●</span>'
        f'<span style="margin:0 6px;">|</span> Database: {DB_CONFIG.get("dbname", "")}'
        f'<span style="margin:0 6px;">|</span> User: {DB_CONFIG.get("user", "")}'
        f'<span style="margin:0 6px;">|</span> Time: {now}</div>'
    )


# ── Sample Prompts ───────────────────────────────────────────────────

_PROMPT_1 = (
    "I have a client in Seattle, Washington, whose property floods after a "
    "neighboring developer regraded their lot and the city modified road drainage during April and May of 2024."
)
_PROMPT_2 = (
    "What is the standard for establishing a prescriptive easement for "
    "drainage across a neighbor's property in Washington State?"
)
_PROMPT_3 = (
    "Can a municipality in Washington be held liable for surface water damage "
    "caused by changes to public road drainage infrastructure?"
)

    
# ── Build the Gradio App ────────────────────────────────────────────

_CSS = """
:root {
    --counsel-bg: #f6f7fb;
    --counsel-panel: #ffffff;
    --counsel-line: #e2e7f3;
    --counsel-text: #1f2340;
    --counsel-muted: #687087;
    --counsel-accent: #5b5ef7;
    --counsel-accent-soft: #eef0ff;
    --counsel-font: Aptos, "Segoe UI", Tahoma, Geneva, Verdana, sans-serif;
}

.gradio-container,
.gradio-container * {
    font-family: var(--counsel-font) !important;
}

.gradio-container {
    background: linear-gradient(180deg, #fbfbfe 0%, #f4f6fb 100%);
}

.counsel-shell {
    background: transparent;
}

.counsel-hdr {
    padding: 2px 2px 10px 2px;
}

.counsel-hdr h1 {
    margin: 0;
    font-size: 24px;
    display: flex;
    align-items: center;
    gap: 10px;
    color: var(--counsel-text);
}

.counsel-hdr p {
    margin: 4px 0 0;
    font-size: 14px;
    color: var(--counsel-muted);
}

.counsel-header-icon {
    width: 30px;
    height: 30px;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    flex: 0 0 30px;
}

.counsel-chat,
.counsel-trace {
    border: 1px solid var(--counsel-line);
    border-radius: 18px;
    background: var(--counsel-panel);
    box-shadow: 0 16px 34px rgba(31, 35, 64, 0.06);
}

.counsel-chat {
    overflow: hidden;
}

.counsel-chatbot,
.counsel-chatbot > .wrap,
.counsel-chatbot .wrap,
.counsel-chatbot .bubble-wrap,
.counsel-chatbot .message-wrap,
.counsel-chatbot .message-row,
.counsel-chatbot .panel,
.counsel-chatbot .scroll-hide {
    background: #ffffff !important;
}

.counsel-chatbot {
    border: 0 !important;
}

.counsel-chatbot .message,
.counsel-chatbot .message.user,
.counsel-chatbot .message.bot,
.counsel-chatbot .bubble {
    background: transparent !important;
    color: var(--counsel-text) !important;
    border: 0 !important;
    box-shadow: none !important;
}

.counsel-chatbot .message-content {
    background: #ffffff !important;
    color: var(--counsel-text) !important;
    border: 1px solid var(--counsel-line) !important;
    box-shadow: none !important;
}

.counsel-chatbot .message.user {
    background: #f4f6ff !important;
}

.counsel-chatbot .message.bot {
    background: #ffffff !important;
}

.counsel-chatbot .message * {
    color: var(--counsel-text) !important;
}

.counsel-chatbot .avatar-container,
.counsel-chatbot .avatar-container * {
    background: #f4f6ff !important;
    color: var(--counsel-accent) !important;
}

.counsel-trace {
    padding: 18px;
}

.counsel-examples {
    align-items: center;
    gap: 8px;
}

.counsel-examples .gr-button {
    border: 1px solid #dde2ff;
    background: #f7f8ff;
    color: #4f56c7;
    border-radius: 10px;
}

.counsel-input-row {
    gap: 12px;
    align-items: center;
}

.counsel-input,
.counsel-input > div,
.counsel-input .wrap,
.counsel-input .block {
    background: #ffffff !important;
    border-radius: 14px !important;
}

.counsel-input textarea,
.counsel-input input,
.counsel-input-row textarea,
.counsel-input-row input {
    background: #ffffff !important;
    border: 1px solid var(--counsel-line) !important;
    border-radius: 14px !important;
    color: var(--counsel-text) !important;
    box-shadow: none !important;
}

.counsel-input textarea::placeholder,
.counsel-input input::placeholder {
    color: #8f96ab !important;
}

.counsel-send {
    border-radius: 14px !important;
}

footer { display: none !important; }
@keyframes spin { from { transform: rotate(0deg); } to { transform: rotate(360deg); } }
.spin { display: inline-block; animation: spin 1s linear infinite; }
"""

with gr.Blocks(title="Counsel AI – Legal Research Assistant", css=_CSS, theme=_UI_THEME) as demo:
    with gr.Column(elem_classes=["counsel-shell"]):
        gr.HTML(
            '<div class="counsel-hdr">'
            f'<h1><span class="counsel-header-icon">{_header_icon_svg()}</span> Counsel AI - Legal Research Assistant</h1>'
            '<p>Ask legal questions. Counsel uses HorizonDB to find, analyze, and synthesize case law.</p>'
            '</div>'
        )

        with gr.Row(equal_height=True):
            with gr.Column(scale=3, elem_classes=["counsel-chat"]):
                chatbot = gr.Chatbot(height=500, show_label=False, type="messages", elem_classes=["counsel-chatbot"])
            with gr.Column(scale=2, elem_classes=["counsel-trace"]):
                trace_panel = gr.HTML(value=_build_trace_html([], 0))

        with gr.Row(elem_classes=["counsel-examples"]):
            gr.HTML('<span style="font-size:13px;color:#687087;padding:0 4px;">Example questions</span>')
            ex1 = gr.Button("Property flooding from neighbor regrading & city drainage", size="sm")
            ex2 = gr.Button("Prescriptive easement for drainage in WA", size="sm")
            ex3 = gr.Button("Municipal liability for road drainage damage", size="sm")

        with gr.Row(elem_classes=["counsel-input-row"]):
            msg = gr.Textbox(
                placeholder="Ask a legal question...",
                show_label=False,
                scale=5,
                container=False,
                elem_classes=["counsel-input"],
            )
            send_btn = gr.Button("Send", variant="primary", scale=1, elem_classes=["counsel-send"])

        gr.HTML(_status_html())

    send_btn.click(_chat_fn, [msg, chatbot], [chatbot, trace_panel, msg])
    msg.submit(_chat_fn, [msg, chatbot], [chatbot, trace_panel, msg])


    ex1.click(lambda: _PROMPT_1, outputs=msg)
    ex2.click(lambda: _PROMPT_2, outputs=msg)
    ex3.click(lambda: _PROMPT_3, outputs=msg)

gr.close_all()
demo.launch(server_port=7860, inline=False)

### 🎉 Congratulations you have completed the Lab!

#### 🚀 Next Steps

We have curated additional resources to enhance your ongoing journey in building AI agents and AI-powered applications with Azure Database for PostgreSQL.

- A more detailed blog post about the legal case example of lab in the [GraphRAG Solution for Azure Database for PostgreSQL](https://aka.ms/pg-graphrag) check the code in the [GitHub repository](https://aka.ms/postgres-graphrag-solution).
- Learn more about [Graph data in Azure Database for PostgreSQL](https://aka.ms/age-blog).
- Get familiar with the new [PostgreSQL extension for Visual Studio Code]().